# Entrenamiento de un modelo YOLOv8 para clasificación de lunares

Si bien el modelo actualmente se enfoca en encontrar y clasificar lunares, en el pipeline del proyecto lo que vamos a utilizar es solamente la funcionalidad de encontrar el area de interes (ROI) para que sea el input de un modelo efficient-netb0.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
!ls /content/drive/MyDrive/Colab/moles_detector_yolov8

data.yaml  README.roboflow.txt	train


In [19]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.7 MB/s eta 0:00:0000:01


El dataset de roboflow no se exporta dividio en train, val, test, por lo que se hace con la herramienta proporcionada por ultralytics (igual requiere un ajuste posterior por la forma de guardar los archivos)

In [ ]:
from ultralytics.data.split import autosplit

autosplit(
    path="/content/drive/MyDrive/Colab/moles_detector_yolov8",
    weights=(0.8, 0.2, 0.0),
    annotated_only=True
)

Autosplitting images from /content/drive/MyDrive/Colab/moles_detector_yolov8, using *.txt labeled images only
: 100% ━━━━━━━━━━━━ 159/159 279.7it/s 0.6s0.1s


### Importante
Para que este split sea utilizado en entreanmiento, hay que cambiar el archivo yaml del dataset para que entre a entrenar las imagenes mediante el archivo txt.
Ejemplo como debería quedar el yaml:
```yaml
train: autosplit_train.txt
val: autosplit_val.txt

nc: 2
names: ['benign', 'malignant']
```
(en un futuro se podrian sacar las clases benign y malignant pero requeriria modificar todos los labels)

In [42]:
!ls drive/MyDrive/Colab

autosplit_train.txt  autosplit_val.txt	moles_detector_yolov8


## Entrenar modelo

In [34]:
from ultralytics import YOLO

In [35]:
# modelo pre-entrenado - para hacer transfer learning con dataset de lunares para marcar el roi
model = YOLO("yolov8n.pt")

In [46]:
!ls /content/drive/MyDrive/Colab/moles_detector_yolov8/

autosplit_train.txt  autosplit_val.txt	data.yaml  train


### Adecuar datos

Debido a que al hacer el split el formato del txt es ./image/.. se cambia al formato del path completo

In [ ]:
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/Colab/moles_detector_yolov8/train")

for txt_file in ["autosplit_train.txt", "autosplit_val.txt"]:
    txt_path = base_dir / txt_file
    if not txt_path.exists():
        txt_path = base_dir.parent / txt_file
    
    lines = txt_path.read_text().strip().splitlines()
    new_lines = []
    for line in lines:
        filename = Path(line.strip()).name
        full_path = base_dir / "images" / filename
        new_lines.append(str(full_path))
    
    txt_path.write_text("\n".join(new_lines) + "\n")
    print(f"Fixed {txt_file}: {len(new_lines)} entries")

Fixed autosplit_train.txt: 117 entries
Fixed autosplit_val.txt: 42 entries


In [48]:
results = model.train(
    data="/content/drive/MyDrive/Colab/moles_detector_yolov8/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    lr0=0.01,
    device=0,
    project="mole_roi",
    name="yolov8n_run1",
    plots=True,
)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab/moles_detector_yolov8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_run14, nbs=64, nms=False, opset=None, optimize=False, optimizer=a

In [49]:
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.0 ms, read: 39.4±9.2 MB/s, size: 48.3 KB)
val: Scanning /content/drive/MyDrive/Colab/moles_detector_yolov8/train/labels.cache... 42 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 42/42 12.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.8s1.1s
                   all         42         42      0.926      0.991      0.984      0.617
                benign         22         22        0.9          1      0.979      0.587
             malignant         20         20      0.952      0.983      0.988      0.646
Speed: 7.1ms preprocess, 8.2ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val
mAP50:    0.9836
mAP50-95: 0.6165
Precision: 0.9257
Recall:    0.9913


In [64]:
! ls ./runs/detect/mole_roi/yolov8n_run14/weights

best.pt  last.pt


Guardar el modelo de forma permanente, fuera de la VM

In [65]:
import shutil

shutil.copy(
    "/content/runs/detect/mole_roi/yolov8n_run14/weights/best.pt",
    "/content/drive/MyDrive/Colab/moles_detector_yolov8/best.pt"
)

'/content/drive/MyDrive/Colab/moles_detector_yolov8/best.pt'